# Compare & validate HOL4 → Lean translations (Gemini 2.5)

This notebook pairs HOL4 theory scripts (`xxxScript.sml`) with the corresponding generated Lean files (`xxx.lean`) and asks **Gemini 2.5** to spot likely translation mistakes.

**Mapping rule**: `xxxScript.sml` (in HOL4) ↔ `xxx.lean` (in Lean).

Paths used:
- Lean: `E:/NUS/mcomp/Dissertation/cakeML/cakeml/lean_type_sound/LeanTypeSound`
- HOL4: `E:/NUS/mcomp/Dissertation/cakeML/cakeml/lean_type_sound/hol4-ultramin`

## Prerequisites

1. Set your Gemini API key in the environment (PowerShell):
   - `$env:GEMINI_API_KEY = "..."`

2. Ensure you have read access to the two folders above.

Notes:
- The notebook writes results to `compare_validate_results.jsonl` in this workspace so you can resume without re-calling the API.
- By default it validates only a small number of pairs; adjust `MAX_FILES` if you want more.

In [19]:
from __future__ import annotations

import os
import json
import time
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

# --- Configuration (edit if needed) ---
LEAN_ROOT = Path(r"E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_type_sound\LeanTypeSound")
HOL4_ROOT = Path(r"E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_type_sound\hol4-ultramin")

# Where to write/append results in this repo
RESULTS_JSONL = Path('compare_validate_results.jsonl')

# Default run size (keep small first; raise as desired)
MAX_FILES = 10

# LLM input guardrails (avoid huge payloads)
MAX_CHARS_PER_FILE = 60_000

# Model choice (override via env var GEMINI_MODEL if desired)
GEMINI_MODEL = os.getenv('GEMINI_MODEL', 'gemini-2.5-pro')

assert LEAN_ROOT.exists(), f'Lean root not found: {LEAN_ROOT}'
assert HOL4_ROOT.exists(), f'HOL4 root not found: {HOL4_ROOT}'

def get_api_key() -> str:
    key = os.getenv('GEMINI_API_KEY')
    if not key:
        raise RuntimeError(
            'GEMINI_API_KEY is not set. In PowerShell run: $env:GEMINI_API_KEY = "..."'
        )
    return key

# --- Import / install google-generativeai ---
try:
    import google.generativeai as genai
except ModuleNotFoundError:
    import sys
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'google-generativeai'])
    import google.generativeai as genai

genai.configure(api_key=get_api_key())
model = genai.GenerativeModel(GEMINI_MODEL)
print('Configured Gemini model:', GEMINI_MODEL)

Configured Gemini model: gemini-2.5-pro


In [21]:
@dataclass(frozen=True)
class FilePair:
    # Canonical pair key (used for caching/results)
    base: str
    hol4_path: Path
    lean_path: Path

def read_text_safely(path: Path) -> str:
    # Prefer UTF-8; replace undecodable bytes to keep pipeline robust
    return path.read_text(encoding='utf-8', errors='replace')

def truncate_middle(text: str, *, max_chars: int) -> str:
    if len(text) <= max_chars:
        return text
    keep_head = max_chars // 2
    keep_tail = max_chars - keep_head
    return (
        text[:keep_head]
        + f"\n\n--- TRUNCATED ({len(text) - max_chars} chars omitted) ---\n\n"
        + text[-keep_tail:]
    )

def find_hol4_scripts(root: Path) -> dict[str, Path]:
    scripts: dict[str, Path] = {}
    for p in root.rglob('*Script.sml'):
        base = p.name.removesuffix('Script.sml')
        key = base.lower()
        # If duplicates exist, keep the shortest path (more 'canonical')
        if key not in scripts or len(str(p)) < len(str(scripts[key])):
            scripts[key] = p
    return scripts

def find_lean_files(root: Path) -> dict[str, Path]:
    leans: dict[str, Path] = {}
    for p in root.rglob('*.lean'):
        key = p.stem.lower()
        if key not in leans or len(str(p)) < len(str(leans[key])):
            leans[key] = p
    return leans

# Special-case mapping: HOL4 base -> Lean base
SPECIAL_MATCHES: dict[str, str] = {
    'lprefix_lub': 'lprefixlub',
    'semantics_min': 'semanticsmin',
}

def collect_pairs(hol4_root: Path, lean_root: Path) -> list[FilePair]:
    hol4 = find_hol4_scripts(hol4_root)
    lean = find_lean_files(lean_root)

    pairs: list[FilePair] = []
    matched_hol4: set[str] = set()
    matched_lean: set[str] = set()

    # 1) Direct name matches
    for b in sorted(set(hol4).intersection(lean)):
        pairs.append(FilePair(base=b, hol4_path=hol4[b], lean_path=lean[b]))
        matched_hol4.add(b)
        matched_lean.add(b)

    # 2) Explicit special-case matches (HOL4 -> Lean)
    for hol4_base, lean_base in SPECIAL_MATCHES.items():
        if hol4_base in hol4 and lean_base in lean:
            if hol4_base in matched_hol4 or lean_base in matched_lean:
                continue
            pairs.append(
                FilePair(base=hol4_base, hol4_path=hol4[hol4_base], lean_path=lean[lean_base])
            )
            matched_hol4.add(hol4_base)
            matched_lean.add(lean_base)

    return pairs

hol4_map = find_hol4_scripts(HOL4_ROOT)
lean_map = find_lean_files(LEAN_ROOT)
pairs = collect_pairs(HOL4_ROOT, LEAN_ROOT)

matched_hol4_paths = {fp.hol4_path for fp in pairs}
matched_lean_paths = {fp.lean_path for fp in pairs}

hol4_unmatched = sorted(
    key for key, path in hol4_map.items()
    if path not in matched_hol4_paths
)
lean_unmatched = sorted(
    key for key, path in lean_map.items()
    if path not in matched_lean_paths
)

print('HOL4 scripts found:', len(hol4_map))
print('Lean files found:', len(lean_map))
print('Pairs found:', len(pairs))
print('Example pairs:')
for fp in pairs[:10]:
    print('-', fp.base)
    print('  HOL4:', fp.hol4_path)
    print('  Lean:', fp.lean_path)

print('\nUnmatched HOL4 scripts:', len(hol4_unmatched))
for key in hol4_unmatched[:20]:
    print('-', key)
    print('  HOL4:', hol4_map[key])

print('\nUnmatched Lean files:', len(lean_unmatched))
for key in lean_unmatched[:20]:
    print('-', key)
    print('  Lean:', lean_map[key])

if len(pairs) == 0:
    # Diagnostics: show a few keys to spot naming mismatches
    print('\nNo pairs found. Sample HOL4 bases:', list(sorted(hol4_map.keys()))[:20])
    print('Sample Lean bases:', list(sorted(lean_map.keys()))[:20])

HOL4 scripts found: 20
Lean files found: 21
Pairs found: 20
Example pairs:
- ast
  HOL4: E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_type_sound\hol4-ultramin\astScript.sml
  Lean: E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_type_sound\LeanTypeSound\Ast.lean
- evaluate
  HOL4: E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_type_sound\hol4-ultramin\evaluateScript.sml
  Lean: E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_type_sound\LeanTypeSound\Evaluate.lean
- evaluateprops
  HOL4: E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_type_sound\hol4-ultramin\evaluatePropsScript.sml
  Lean: E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_type_sound\LeanTypeSound\EvaluateProps.lean
- ffi
  HOL4: E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_type_sound\hol4-ultramin\ffiScript.sml
  Lean: E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_type_sound\LeanTypeSound\Ffi.lean
- fpsem
  HOL4: E:\NUS\mcomp\Dissertation\cakeML\cakeml\lean_type_sound\hol4-ultramin\fpSemScript.sml
  Lean: E:\NUS\mcomp\Dissertation\cake

In [22]:
def load_existing_results(path: Path) -> dict[str, dict]:
    if not path.exists():
        return {}
    existing: dict[str, dict] = {}
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            key = obj.get('base')
            if key:
                existing[key] = obj
    return existing

existing = load_existing_results(RESULTS_JSONL)
print('Existing cached results:', len(existing))

# Choose which pairs to run this session
todo = [p for p in pairs if p.base not in existing]
print('Pairs remaining:', len(todo))

# Select a small batch by default (edit MAX_FILES above)
to_run = todo[:MAX_FILES]
print('Pairs selected for this run:', len(to_run))
print([p.base for p in to_run])

Existing cached results: 0
Pairs remaining: 20
Pairs selected for this run: 10
['ast', 'evaluate', 'evaluateprops', 'ffi', 'fpsem', 'location', 'misc', 'mllist', 'mlstring', 'namespace']


In [23]:
SYSTEM_INSTRUCTIONS = """\
You are reviewing a translation from HOL4/CakeML SML theory scripts to Lean 4 code.

Goal: identify likely translation errors, based only on the supplied texts. If you believe the translation is correct, return "good translation", otherwise provide detailed feedback on what is wrong.

Constraints: you may be uncertain because the translation is not mechanically verified; be explicit about uncertainty.

Return STRICT JSON only (no markdown), matching the schema described in the user prompt.
"""

SCHEMA_TEXT = """{
  \"base\": string,
  \"verdict\": \"ok\" | \"suspect\" | \"wrong\" | \"uncertain\",
  \"confidence\": number,
  \"summary\": string,
  \"issues\": [{
    \"category\": string,
    \"severity\": \"low\" | \"medium\" | \"high\",
    \"evidence\": string,
    \"suggested_fix\": string
  }],
  \"notes\": string
}"""

def build_prompt(*, base: str, hol4_path: Path, lean_path: Path, hol4_text: str, lean_text: str) -> str:
    return f"""\
File pair base name: {base}

HOL4 source: {hol4_path}
Lean target: {lean_path}

Task: Compare the HOL4 script and the Lean file. Look for translation mistakes.

Please produce STRICT JSON with this schema:
{SCHEMA_TEXT}

Important:
- Only output JSON.
- If you claim something is wrong, include concrete evidence from the provided texts.
- If unsure, use verdict \"uncertain\" or \"suspect\".

--- HOL4 (possibly truncated) ---
{hol4_text}

--- Lean (possibly truncated) ---
{lean_text}
"""

def call_gemini_json(prompt: str, *, max_retries: int = 4) -> dict:
    # Basic retry with jitter (rate limits / transient errors)
    last_err: Optional[Exception] = None
    for attempt in range(max_retries):
        try:
            resp = model.generate_content(
                [SYSTEM_INSTRUCTIONS, prompt],
                generation_config={
                    'temperature': 0.1,
                    'top_p': 0.95,
                    'max_output_tokens': 2048,
                },
            )
            text = (resp.text or '').strip()
            # Parse strict JSON; if model wrapped it, extract first JSON object.
            try:
                return json.loads(text)
            except json.JSONDecodeError:
                start = text.find('{')
                end = text.rfind('}')
                if start != -1 and end != -1 and end > start:
                    return json.loads(text[start : end + 1])
                raise
        except Exception as e:
            last_err = e
            sleep_s = (2 ** attempt) + random.uniform(0.0, 0.5)
            time.sleep(sleep_s)
    raise RuntimeError(f'Gemini call failed after retries: {last_err}')

In [24]:
def analyze_pair(fp: FilePair) -> dict:
    hol4_raw = read_text_safely(fp.hol4_path)
    lean_raw = read_text_safely(fp.lean_path)

    hol4_text = truncate_middle(hol4_raw, max_chars=MAX_CHARS_PER_FILE)
    lean_text = truncate_middle(lean_raw, max_chars=MAX_CHARS_PER_FILE)

    prompt = build_prompt(
        base=fp.base,
        hol4_path=fp.hol4_path,
        lean_path=fp.lean_path,
        hol4_text=hol4_text,
        lean_text=lean_text,
    )

    result = call_gemini_json(prompt)
    if isinstance(result, dict):
        result.setdefault('base', fp.base)
        result.setdefault('hol4_path', str(fp.hol4_path))
        result.setdefault('lean_path', str(fp.lean_path))
    return result

def append_jsonl(path: Path, obj: dict) -> None:
    with path.open('a', encoding='utf-8') as f:
        f.write(json.dumps(obj, ensure_ascii=False))
        f.write('\n')

# Run validation (small batch by default)
for i, fp in enumerate(to_run, start=1):
    print(f'[{i}/{len(to_run)}] {fp.base}')
    try:
        out = analyze_pair(fp)
        append_jsonl(RESULTS_JSONL, out)
        # Gentle pacing (avoid burst rate limiting)
        time.sleep(0.6)
    except Exception as e:
        err_obj = {
            'base': fp.base,
            'verdict': 'uncertain',
            'confidence': 0.0,
            'summary': 'Error during analysis call',
            'issues': [],
            'notes': str(e),
            'hol4_path': str(fp.hol4_path),
            'lean_path': str(fp.lean_path),
        }
        append_jsonl(RESULTS_JSONL, err_obj)
        print('  ERROR:', e)
        time.sleep(1.0)

print('Done. Results appended to:', RESULTS_JSONL.resolve())

[1/10] ast
  ERROR: Gemini call failed after retries: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
[2/10] evaluate


KeyboardInterrupt: 

In [ ]:
# Summarize results
try:
    import pandas as pd
except ModuleNotFoundError:
    import sys, subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas'])
    import pandas as pd

rows = list(load_existing_results(RESULTS_JSONL).values())
df = pd.DataFrame(rows)
if len(df) == 0:
    print('No results yet.')
else:
    cols = [c for c in ['base', 'verdict', 'confidence', 'summary', 'hol4_path', 'lean_path'] if c in df.columns]
    display(df[cols].sort_values(['verdict', 'confidence'], ascending=[True, True]).head(50))
    display(df['verdict'].value_counts(dropna=False))